In [1]:
!pip install tensorflow
!pip install scikit-image


In [2]:
from keras.layers import Conv2D, UpSampling2D, InputLayer, Conv2DTranspose
from keras.layers import Activation, Dense, Dropout, Flatten
from keras.layers import BatchNormalization
from keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from skimage.color import rgb2lab, lab2rgb, rgb2gray, xyz2lab
from skimage.io import imsave
import numpy as np
import os
import random
import tensorflow as tf

C:\Users\ruthw_8xj55dr\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [3]:
# Get images
image = img_to_array(load_img('RKK.jpg'))
image = np.array(image, dtype=float)

In [4]:
from skimage.color import rgb2lab
import numpy as np

# Get the dimensions of the image
height, width, _ = image.shape
shorter_dim = min(height, width)

# Calculate cropping coordinates to get the center square
start_x = (width - shorter_dim) // 2
start_y = (height - shorter_dim) // 2

# Crop the center square
image_cropped = image[start_y:start_y + shorter_dim, start_x:start_x + shorter_dim]

# Now resize the cropped image to 400x400 for the model
from skimage.transform import resize
image_resized = resize(image_cropped, (400, 400), anti_aliasing=True)

# Convert to LAB and split channels
X = rgb2lab(1.0 / 255 * image_resized)[:, :, 0]      # Lightness channel
Y = rgb2lab(1.0 / 255 * image_resized)[:, :, 1:]     # AB channels
Y /= 128

# Reshape to match model input requirements
X = X.reshape(1, 400, 400, 1)
Y = Y.reshape(1, 400, 400, 2)


In [15]:
#from skimage.transform import resize

# Assuming `image` is your input image in RGB format
#image_resized = resize(image, (400, 400), anti_aliasing=True)

# Convert to LAB and split channels
#X = rgb2lab(1.0 / 255 * image_resized)[:, :, 0]      # Lightness channel
#Y = rgb2lab(1.0 / 255 * image_resized)[:, :, 1:]     # AB channels
#Y /= 128

# Reshape to match model input requirements
#X = X.reshape(1, 400, 400, 1)
#Y = Y.reshape(1, 400, 400, 2)


In [28]:
#X = rgb2lab(1.0/255*image)[:,:,0]
#Y = rgb2lab(1.0/255*image)[:,:,1:]
#Y /= 128
#X = X.reshape(1, 400, 400, 1)
#Y = Y.reshape(1, 400, 400, 2)

ValueError: cannot reshape array of size 8629248 into shape (1,400,400,1)

In [5]:
# Building the neural network
model = Sequential()
model.add(InputLayer(input_shape=(None, None, 1)))
model.add(Conv2D(8, (3, 3), activation='relu', padding='same', strides=2))
model.add(Conv2D(8, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(16, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(16, (3, 3), activation='relu', padding='same', strides=2))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same', strides=2))
model.add(UpSampling2D((2, 2)))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(UpSampling2D((2, 2)))
model.add(Conv2D(16, (3, 3), activation='relu', padding='same'))
model.add(UpSampling2D((2, 2)))
model.add(Conv2D(2, (3, 3), activation='tanh', padding='same'))

C:\Users\ruthw_8xj55dr\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [6]:
# Finish model
model.compile(optimizer='rmsprop',loss='mse')

In [7]:
model.fit(x=X,
    y=Y,
    batch_size=1,
    epochs=10)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.1198
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.0188
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0058
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0039
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0033
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0030
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0030
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0029
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0029
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.0028


In [8]:
# Assuming `model` is already trained, and `X` is your input data
output = model.predict(X)
output *= 128  # Scale the output as per LAB color space requirements

# Convert output to image format and save
from skimage import img_as_ubyte

# Output colorizations
cur = np.zeros((400, 400, 3))
cur[:, :, 0] = X[0][:, :, 0]  # Lightness channel from the input
cur[:, :, 1:] = output[0]     # Predicted color channels

# Convert to uint8 before saving
cur_uint8 = img_as_ubyte(lab2rgb(cur))
imsave("img_result.png", cur_uint8)

# Convert grayscale version to uint8 and save
gray_version = img_as_ubyte(rgb2gray(lab2rgb(cur)))
imsave("img_gray_version.png", gray_version)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step


In [8]:
#from skimage import img_as_ubyte

# Output colorizations
#cur = np.zeros((400, 400, 3))
#cur[:, :, 0] = X[0][:, :, 0]
#cur[:, :, 1:] = output[0]

# Convert to uint8 before saving
#cur_uint8 = img_as_ubyte(lab2rgb(cur))
#imsave("img_result.png", cur_uint8)

# Convert grayscale version to uint8 and save
#gray_version = img_as_ubyte(rgb2gray(lab2rgb(cur)))
#imsave("img_gray_version.png", gray_version)


NameError: name 'output' is not defined